In [14]:
print("hey")

hey


In [15]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [16]:
from langchain_groq import ChatGroq
import os

In [17]:
import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'meta-llama/llama-prompt-guard-2-86m', 'object': 'model', 'created': 1748632165, 'owned_by': 'Meta', 'active': True, 'context_window': 512, 'public_apps': None, 'max_completion_tokens': 512, 'name': 'Prompt Guard 2 86M', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 512, 'max_output_length': 512, 'pricing': {'prompt': '0.00000004', 'completion': '0.00000004', 'image': '0', 'request': '0', 'input_cache_read': '0.00000002'}, 'supported_sampling_parameters': ['temperature', 'top_p', 'stop', 'seed', 'max_tokens'], 'supported_features': ['json_mode']}, {'id': 'meta-llama/llama-prompt-guard-2-22m', 'object': 'model', 'created': 1748632101, 'owned_by': 'Meta', 'active': True, 'context_window': 512, 'public_apps': None, 'max_completion_tokens': 512, 'name': 'Llama Prompt Guard 2 22M', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 512, 'max_output_length': 512, 'pricing': {'prompt': '0.00000003'

In [18]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [19]:
llm.invoke("What is the llm?")

AIMessage(content='**LLM** stands for **Large Language Model**.  \n\nA large language model is a type of artificial‑intelligence system that has been trained on massive amounts of text data (often billions of words) to learn the statistical patterns of language. By processing this data, the model develops the ability to:\n\n1. **Generate text** – write sentences, paragraphs, or even full articles that read like they were written by a human.  \n2. **Understand prompts** – interpret questions, instructions, or any input text and produce a relevant response.  \n3. **Perform many language tasks** – translation, summarization, sentiment analysis, question answering, code generation, and more—often without needing task‑specific fine‑tuning.\n\n### How It Works (in a nutshell)\n\n| Step | What Happens |\n|------|--------------|\n| **Training** | The model processes huge corpora of text, adjusting billions of parameters (weights) to predict the next word in a sequence. |\n| **Architecture** | 

In [20]:
from langchain.tools import tool

@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

In [21]:
search_database.invoke({
    "query": "customer name",
    "limit": 5
})

"Found 5 results for 'customer name'"

In [22]:
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_openai import ChatOpenAI

# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"

# Update memory
@tool
def save_user_info(user_id: str, name: str, age: int, email: str, runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(("users",), user_id, {"name": name, "age": age, "email": email})
    return "Successfully saved user info."


store = InMemoryStore()
agent = create_agent(
    llm,
    tools=[get_user_info, save_user_info],
    store=store
)

# First session: save user info
agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})



{'messages': [HumanMessage(content='Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev', additional_kwargs={}, response_metadata={}, id='86f85377-0484-46e7-ae97-f9cee1b170e4'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants to save user info. Use save_user_info function.', 'tool_calls': [{'id': 'fc_3afa8fbe-e339-4601-9bed-5a36c2b4f34b', 'function': {'arguments': '{"age":25,"email":"foo@langchain.dev","name":"Foo","user_id":"abc123"}', 'name': 'save_user_info'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 179, 'total_tokens': 248, 'completion_time': 0.145127594, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.045087919, 'prompt_tokens_details': None, 'queue_time': 0.295276146, 'total_time': 0.190215513}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c800245357', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logpro

In [23]:
# Second session: get user info
agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})
# Here is the user info for user with ID "abc123":
# - Name: Foo
# - Age: 25
# - Email: foo@langchain.dev

{'messages': [HumanMessage(content="Get user info for user with id 'abc123'", additional_kwargs={}, response_metadata={}, id='211653d8-c72b-4fb3-bc2c-355ea45b472e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to call get_user_info function with user_id 'abc123'.", 'tool_calls': [{'id': 'fc_c1f3064b-9859-40f4-9c9f-bcaabba8605c', 'function': {'arguments': '{"user_id":"abc123"}', 'name': 'get_user_info'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 164, 'total_tokens': 210, 'completion_time': 0.096551266, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.006965205, 'prompt_tokens_details': None, 'queue_time': 0.317369243, 'total_time': 0.103516471}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fe269835c7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1ce96-4a4d-7042-ab1d-466ad78d8cc8-0', too

In [24]:

import requests
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool


class WeatherInput(BaseModel):
    city: str = Field(description="City name")


def get_weather(city: str) -> str:
    """Fetch current weather for a city."""

    # Step 1: Convert city name to latitude and longitude
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_response = requests.get(
        geo_url,
        params={"name": city, "count": 1, "language": "en"},
        timeout=10,
    )
    geo_response.raise_for_status()
    places = geo_response.json().get("results", [])

    if not places:
        return f"City not found: {city}"

    place = places[0]
    latitude = place["latitude"]
    longitude = place["longitude"]

    # Step 2: Fetch current weather
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_response = requests.get(
        weather_url,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "current": (
                "temperature_2m,"
                "relative_humidity_2m,"
                "apparent_temperature,"
                "wind_speed_10m"
            ),
            "timezone": "auto",
        },
        timeout=10,
    )
    weather_response.raise_for_status()
    current = weather_response.json()["current"]

    return (
        f"Weather in {place['name']}: "
        f"Temperature: {current['temperature_2m']} °C, "
        f"Feels like: {current['apparent_temperature']} °C, "
        f"Humidity: {current['relative_humidity_2m']}%, "
        f"Wind speed: {current['wind_speed_10m']} km/h, "
        f"Observation time: {current['time']}"
    )


weather_tool = StructuredTool.from_function(
    func=get_weather,
    name="get_weather",
    description="Fetch current weather data for a city",
    args_schema=WeatherInput,
)

# Test the tool directly
print(weather_tool.invoke({"city": "Kozhikode"}))


Weather in Kozhikode: Temperature: 31.6 °C, Feels like: 37.6 °C, Humidity: 60%, Wind speed: 8.2 km/h, Observation time: 2026-11-12T12:30


In [25]:

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[weather_tool],
    system_prompt="You are a helpful weather assistant."
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the weather in Kozhikode?"
        }
    ]
})

print(result["messages"][-1].content)


Here’s the current weather for **Kozhikode**:

- **Temperature:** 31.6 °C  
- **Feels like:** 37.6 °C  
- **Humidity:** 60 %  
- **Wind speed:** 8.2 km/h  
- **Observation time:** 2026‑11‑12 12:30 (local time)  


In [38]:
import requests

class WeatherForecastTool:
    def __init__(self):
        self.api_key = os.environ.get("OPENWEATHERMAP_API_KEY")
        self.base_url = "https://api.openweathermap.org/data/2.5"
        self.geourl="https://api.openweathermap.org/geo/1.0/direct"

    def get_current_weather(self, place:str):
        """Get current weather using coordinates."""

        geo_url=f"{self.geourl}"

        geo_params = {
            "q": place,
            "appid": self.api_key}
        
        geo_response = requests.get(geo_url, params=geo_params, timeout=10)
        geo_response.raise_for_status()
        geo_data = geo_response.json()[0]  # Get the first result
        base_url = f"{self.base_url}/weather"
       
       
        params = {
            "lat": geo_data["lat"],
            "lon": geo_data["lon"],
            "appid": self.api_key,
            "units": "metric",
        }

        response = requests.get(base_url, params=params, timeout=10)
        response.raise_for_status()

        return response.json()
    
    def get_forecast_weather(self, place:str, days:int=5):
        """Get weather forecast of a place"""
        cnt=days*8  # OpenWeatherMap provides data in 3-hour intervals, so 8 intervals per day
        try:
            url = f"{self.base_url}/forecast"
            params = {
                "q": place,
                "appid": self.api_key,
                "cnt": cnt,
                "units": "metric"
            }
            response = requests.get(url, params=params)
            return response.json() if response.status_code == 200 else {}
        except Exception as e:
            raise e

In [39]:
weather_data = WeatherForecastTool().get_current_weather(place="london")
weather_data

{'coord': {'lon': -0.1278, 'lat': 51.5074},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01d'}],
 'base': 'stations',
 'main': {'temp': 4.66,
  'feels_like': 2.44,
  'temp_min': 3.58,
  'temp_max': 5.18,
  'pressure': 1005,
  'humidity': 90,
  'sea_level': 1005,
  'grnd_level': 1003},
 'visibility': 10000,
 'wind': {'speed': 2.57, 'deg': 350},
 'clouds': {'all': 2},
 'dt': 1794474390,
 'sys': {'type': 2,
  'id': 2075535,
  'country': 'GB',
  'sunrise': 1794467585,
  'sunset': 1794500180},
 'timezone': 0,
 'id': 2643743,
 'name': 'London',
 'cod': 200}

In [40]:
forcast_data = WeatherForecastTool().get_forecast_weather(place="london")
forcast_data

{'cod': '200',
 'message': 0,
 'cnt': 40,
 'list': [{'dt': 1794484800,
   'main': {'temp': 5.79,
    'feels_like': 3.07,
    'temp_min': 5.79,
    'temp_max': 8.04,
    'pressure': 1005,
    'sea_level': 1005,
    'grnd_level': 1003,
    'humidity': 85,
    'temp_kf': -2.25,
    'dew_point': 3.47},
   'weather': [{'id': 800,
     'main': 'Clear',
     'description': 'clear sky',
     'icon': '01d'}],
   'clouds': {'all': 1},
   'wind': {'speed': 3.58, 'deg': 2, 'gust': 8.72},
   'visibility': 10000,
   'pop': 0,
   'sys': {'pod': 'd'},
   'dt_txt': '2026-11-12 12:00:00'},
  {'dt': 1794495600,
   'main': {'temp': 6.88,
    'feels_like': 4.64,
    'temp_min': 6.88,
    'temp_max': 7.99,
    'pressure': 1006,
    'sea_level': 1006,
    'grnd_level': 1004,
    'humidity': 74,
    'temp_kf': -1.11,
    'dew_point': 2.57},
   'weather': [{'id': 800,
     'main': 'Clear',
     'description': 'clear sky',
     'icon': '01d'}],
   'clouds': {'all': 1},
   'wind': {'speed': 3.19, 'deg': 356, 'gu

In [41]:

if forcast_data:
    forecasts = forcast_data.get("list", [])
    forecast_str = f"Weather forecast in london for the next  days:\n"
    for forecast in forecasts:
        date = forecast.get('dt_txt', 'N/A')
        temp = forecast.get('main', {}).get('temp', 'N/A')

        desc = forecast.get('weather', [{}])[0].get('description', 'N/A')
        forecast_str += f"{date}: {temp}°C, {desc}\n"
        
    print(forecast_str)

        
        

Weather forecast in london for the next  days:
2026-11-12 12:00:00: 5.79°C, clear sky
2026-11-12 15:00:00: 6.88°C, clear sky
2026-11-12 18:00:00: 5.78°C, clear sky
2026-11-12 21:00:00: 4.83°C, clear sky
2026-11-13 00:00:00: 3.24°C, clear sky
2026-11-13 03:00:00: 2.23°C, clear sky
2026-11-13 06:00:00: 1.46°C, clear sky
2026-11-13 09:00:00: 3.88°C, clear sky
2026-11-13 12:00:00: 7.82°C, clear sky
2026-11-13 15:00:00: 8.5°C, clear sky
2026-11-13 18:00:00: 4.14°C, clear sky
2026-11-13 21:00:00: 2.49°C, clear sky
2026-11-14 00:00:00: 1.8°C, few clouds
2026-11-14 03:00:00: -0.64°C, overcast clouds
2026-11-14 06:00:00: -1.57°C, overcast clouds
2026-11-14 09:00:00: -0.43°C, overcast clouds
2026-11-14 12:00:00: 6.83°C, few clouds
2026-11-14 15:00:00: 8.04°C, overcast clouds
2026-11-14 18:00:00: 5.14°C, overcast clouds
2026-11-14 21:00:00: 4.6°C, overcast clouds
2026-11-15 00:00:00: 4.98°C, overcast clouds
2026-11-15 03:00:00: 5.15°C, broken clouds
2026-11-15 06:00:00: 6.85°C, broken clouds
2026

In [42]:
from collections import defaultdict

In [43]:
def format_forecast_data(forecast_data):
    """Format the forecast data into a more readable structure."""
    daily = defaultdict(list)
    
    for forecast in forecast_data.get("list", []):
        date = forecast.get('dt_txt', 'N/A').split(' ')[0]  # Get only the date part
        daily[date].append(forecast)

    result = {
        "data": []
    }

    for date, forecasts in daily.items():
        temps = [f.get('main', {}).get('temp', 'N/A') for f in forecasts]
        descriptions = [f.get('weather', [{}])[0].get('description', 'N/A') for f in forecasts]
        description=max(set(descriptions), key=descriptions.count)  # Most common description

        result["data"].append({
            "date": date,
            "min_temp":f'{min(temps) if temps else None}°C',
            "max_temp": f'{max(temps) if temps else None}°C',
            "description": description
        })

    return json.dumps(result, indent=2, ensure_ascii=False)

In [48]:
import json
formated_forecast = format_forecast_data(forcast_data)


In [45]:
formated_forecast = json.loads(formated_forecast)
if formated_forecast:
    forecast_str = "Weather forecast in London for the next 5 days:\n"

    for data in formated_forecast["data"]:
        date = data["date"]
        min_temp = data["min_temp"]
        max_temp = data["max_temp"]
        desc = data["description"]

        forecast_str += (
            f"{date}: Min {min_temp}, Max {max_temp}, "
            f"{desc}\n"
        )

    print(forecast_str)


Weather forecast in London for the next 5 days:
2026-11-12: Min 4.83°C, Max 6.88°C, clear sky
2026-11-13: Min 1.46°C, Max 8.5°C, clear sky
2026-11-14: Min -1.57°C, Max 8.04°C, overcast clouds
2026-11-15: Min 4.98°C, Max 11.15°C, light rain
2026-11-16: Min 3.55°C, Max 9.68°C, overcast clouds
2026-11-17: Min 0°C, Max 2.29°C, few clouds



In [34]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

True

In [35]:
import os
import requests

class CurrencyConverterTool:

    def __init__(self):
        self.api_key = os.environ.get("EXCHANGE_RATE_API_KEY")
        self.base_url = "https://api.exchangerate.host/convert"

    def convert_currency(
        self,
        amount: float,
        from_currency: str,
        to_currency: str
    ) -> float:

        params = {
            "access_key": self.api_key,
            "from": from_currency,
            "to": to_currency,
            "amount": amount
        }

        response = requests.get(self.base_url, params=params)
        print(response.url)
        response.raise_for_status()
        print(f"Response status code: {response.status_code}")
        data = response.json()
        print("data:", data)
        if data.get("success"):
            result = data["result"]

            print(
                f"{amount} {from_currency} = "
                f"{result:.2f} {to_currency}"
            )

            print(f"Rate used: {data['info']['quote']}")

            return result

        raise ValueError(data.get("error", {}).get("info", "Conversion failed"))

In [36]:
CurrencyConverterTool().convert_currency(amount=100.0, from_currency="USD", to_currency="INR")

https://docs.apilayer.com/exchangerate/docs/api-documentation
Response status code: 200


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
import os
import requests

In [ ]:
api_key = os.getenv("EXCHANGE_RATE_API_KEY")

In [ ]:
api_key

'ba4f3bbd030514759be705de'

In [ ]:
import requests

class CurrencyConverter:
    def __init__(self, api_key: str):
        self.base_url = f"https://v6.exchangerate-api.com/v6/{api_key}/latest/"
    
    def convert_currency(self, amount:float, from_currency:str, to_currency:str):
        """Convert the amount from one currency to another"""
        url = f"{self.base_url}/{from_currency}"
        response = requests.get(url)
        if response.status_code != 200:
            raise Exception("API call failed:", response.json())
        rates = response.json()["conversion_rates"]
        if to_currency not in rates:
            raise ValueError(f"{to_currency} not found in exchange rates.")
        return amount * rates[to_currency]

In [ ]:
CurrencyConverter(api_key=api_key).convert_currency(amount=100.0, from_currency="USD", to_currency="INR")


9394.880000000001

In [ ]:
import pytest
from utils.currency_converter import CurrencyConverter
import os
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("EXCHANGE_RATE_API_KEY") is not None, "EXCHANGE_RATE_API_KEY environment variable is not set."


def test_real_currency_conversion():

    converter = CurrencyConverter(
        api_key=os.environ.get("EXCHANGE_RATE_API_KEY")
    )

    result = converter.convert(amount=100.0, from_currency="USD", to_currency="INR")

   

    print("Converted amount:", result)